In [1]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class DualRecommenderEngine:
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        self.tfidf_matrix = None
        self.transformer_embeddings = None

    def train_tfidf(self, text_column:str = 'metadata', max_features:int = 10000):
        self.tfidf = TfidfVectorizer(stop_words="english", max_features=max_features)
        self.tfidf_matrix = self.tfidf.fit_transform(self.df[text_column])
        print("TF-IDF training is complete.")

    def train_transformer(self, text_column:str = "metadata",
                          model_name:str = "nomic-ai/nomic-embed-text-v1.5"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.transformer_model = SentenceTransformer(model_name, trust_remote_code=True).to(self.device)
        self.transformer_model.max_seq_length = 1024

        self.transformer_embeddings = self.transformer_model.encode(self.df[text_column].to_list(),
                                      convert_to_tensor=True, show_progress_bar=True,
                                      prompt_name='document',
                                      batch_size=16)
        print('Transformer training is complete.')

    def get_recommendations(self, asin:str, top_k:int = 5) -> dict:

        if asin not in self.df['parent_asin'].values:
            return {"error": "ASIN not found in dataset."}

        idx = self.df.index[self.df['parent_asin'] == asin].to_list()[0]

        # TFIDF retrieval
        sim_scores_tfidf = cosine_similarity(self.tfidf_matrix[idx], self.tfidf_matrix).flatten()
        top_scores_tfidf = sim_scores_tfidf.argsort()[-(top_k+1):-1][::-1]

        #Transformer retrieval
        sim_scores_trans = util.cos_sim(self.transformer_embeddings[idx], self.transformer_embeddings)[0]
        top_scores_trans = torch.topk(sim_scores_trans, k=top_k+1).indices[1:].cpu().numpy()

        return {
            'target_asin': asin,
            'target_title': self.df.iloc[idx]['title'],
            'TFIDF_recs': self._format_results(top_scores_tfidf),
            'Transformer_recs': self._format_results(top_scores_trans)
        }

    def _format_results(self, indices) -> list:
        results = []
        for i in indices:
            price = self.df.iloc[i]['price']
            results.append({
                'asin': self.df.iloc[i]['parent_asin'],
                'title': self.df.iloc[i]['title'],
                'price': "unknown" if price == -1 else f"${price:.2f}"
            })
        return results

In [2]:
df = pd.read_csv('/kaggle/input/datasets/tahamah01/amazon-reviews-dataset-for-cbf-recommender/cbf_data.csv')
recommender = DualRecommenderEngine(df)
df.head()

,Unnamed: 0,parent_asin,title,price,average_rating,metadata
0,0,B013SK1JTY,ARAREE Slim Diary Cell Phone Case for Samsung ...,-1.00,3.8,"araree cell phones & accessories cases, holste..."
1,1,B07ZPSG8P5,Bastmei for OnePlus 7T Case Extremely Light Ul...,11.98,4.4,"bastmei cell phones & accessories cases, holst..."
2,2,B00GKR3L12,Wireless Fones Branded New Iphone 5C/LITE Hot ...,-1.00,4.0,wireless fones cell phones & accessories iphon...
3,3,B00PB8U8BW,"iPhone 6 Plus + Case, DandyCase Perfect PATTER...",-1.00,4.0,dandycase cell phones & accessories iphone acc...
4,4,B07D3RHSRV,"Case for Galaxy S6/S6 Edge, Thin Translucent V...",-1.00,4.0,"7pite cell phones & accessories cases, holster..."


In [3]:
recommender.train_tfidf(text_column="metadata")

TF-IDF training is complete.


In [4]:
recommender.train_transformer(text_column="metadata")

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Transformer training is complete.


In [5]:
recommender.get_recommendations('B00GKR3L12')

{'target_asin': 'B00GKR3L12',
 'target_title': 'Wireless Fones Branded New Iphone 5C/LITE Hot Pink Lace detail Leather Wallet Folio Case',
 'TFIDF_recs': [{'asin': 'B00KDQN7HY',
   'title': 'Wirless Fones TM Samsung Galaxy S5 Case Dual Layer Hybrid Impact Resistant Protective Case Circular Black and White Flower Snap on over Teal Skin With Wireless Fones Logo Wrist Band+ Pry Tool + Screen Protector',
   'price': 'unknown'},
  {'asin': 'B00QZHXKRM',
   'title': 'Wireless Fones TM Nokia Lumia 1520 TUFF IMPACT HYBRID Cover Case Black & Silver Chevron Bling + Black Silicone (Wireless Fones TM Wristband Included)',
   'price': 'unknown'},
  {'asin': 'B01B4HHIJ8',
   'title': 'For iPhone 5C,iPhone 5C Case,iPhone 5C Cases,iPhone 5C Leather Case,iPhone 5C Wallet Case,Case for iPhone 5C,Leather 5C Case,Addigital Wallet Cover for iPhone 5C for Girls for Boys #08',
   'price': 'unknown'},
  {'asin': 'B00ZPBLPG2',
   'title': 'mattWill Deluxe Anti-Scratch Hard Shell Polycarbonate Hard Case for iPh

In [6]:
import joblib
import torch

joblib.dump((recommender.tfidf_matrix, recommender.tfidf), 'tfidf_matrix.joblib', compress=3)

torch.save(recommender.transformer_embeddings, "transformer_vectors.pt")